# MiniCells Core Validation 002B — Sparse Functional Write Assemblies

Formal run for the frozen `core-validation-002b` protocol. Core Validation 002 remains `WRITE_ADDRESSABILITY_NOT_SUPPORTED`; this is a new test of sparse functional subspace write addresses.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

BRANCH = 'codex/core-validation-002b-sparse-write-assembly'
REPO = 'https://github.com/ArcheLabs/mini-cells.git'
ROOT = Path('/kaggle/working/mini-cells')
if not ROOT.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO,str(ROOT)], check=True)
else:
    subprocess.run(['git','fetch','origin',BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git','checkout',BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git','reset','--hard',f'origin/{BRANCH}'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('HEAD', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('TREE', subprocess.check_output(['git','rev-parse','HEAD^{tree}'], text=True).strip())


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev]'], check=True)
subprocess.run([sys.executable,'-m','pytest','-q','tests/research/04-continual-learning-core/test_core_validation_002.py','tests/research/04-continual-learning-core/test_core_validation_002b.py'], check=True)
subprocess.run([sys.executable,'scripts/research/run_core_validation_002b.py','--smoke','--device','cpu'], check=True)
print('002/002B tests and 002B CPU smoke passed.')


In [ ]:
import torch
print({'torch':torch.__version__,'cuda':torch.version.cuda,'gpu_count':torch.cuda.device_count(),'gpus':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
assert torch.cuda.is_available(), 'Formal Core Validation 002B requires a CUDA GPU.'


In [ ]:
OUT = ROOT / 'results' / 'core-validation-002b-sparse-write-assembly'
if OUT.exists():
    import shutil
    shutil.rmtree(OUT)
subprocess.run([sys.executable,'scripts/research/run_core_validation_002b.py','--device','cuda'], check=True)
subprocess.run([sys.executable,'scripts/research/report_core_validation_002b.py'], check=True)


In [ ]:
decision = json.loads((OUT/'decision.json').read_text())
print(json.dumps(decision, indent=2, sort_keys=True))
import pandas as pd
display(pd.read_csv(OUT/'gate-summary.csv'))
display(pd.read_csv(OUT/'seed-summary.csv').query("variant_kind in ['assembly','global_ridge']"))


In [ ]:
PUBLISH_RESULTS = True
if PUBLISH_RESULTS:
    command = [sys.executable,'scripts/research/publish_core_validation_002b.py','--push']
    version = os.environ.get('KAGGLE_SCRIPT_VERSION_ID')
    if version:
        command += ['--kaggle-script-version-id', version]
    subprocess.run(command, check=True)
else:
    print('Formal results left in', OUT)
